In [1]:
from core import (
    skf,
    X_train,
    Y_train,
    evaluate_model,
    save_model,
    save_results,
    CAT_FEATURES,
    score
)

from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV


cat = CatBoostClassifier(
    random_state=42,
    verbose=0,
    thread_count=-1,
    eval_metric='AUC',
    od_type='Iter',
    od_wait=50,
    auto_class_weights='Balanced'
)

param_dist_cat = {
    'iterations': [200, 500, 800, 1000, 1500],
    'depth': [1, 2, 3, 4, 5, 6],
    'learning_rate': [0.002, 0.005, 0.008, 0.01, 0.02],
    'l2_leaf_reg': [1, 3, 5, 7],
    'bagging_temperature': [0, 0.5, 1, 1.5, 2],
    'random_strength': [3, 5, 8],
    'border_count': [32, 64, 128],
    'grow_policy': ['SymmetricTree', 'Depthwise', 'Lossguide'],
    'min_data_in_leaf': [1, 3, 5, 10],
}

random_cat = RandomizedSearchCV(
    estimator=cat,
    param_distributions=param_dist_cat,
    n_iter=35,
    cv=skf,
    scoring='roc_auc',
    n_jobs=1,
    random_state=42,
    verbose=1
)

random_cat.fit(X_train, Y_train, cat_features=CAT_FEATURES)

best_cat = random_cat.best_estimator_

print(f'Лучшие параметры CatBoost: {random_cat.best_params_}')
print(f'Лучшая ROC_AUC: {random_cat.best_score_:.4f}')

results_cat = evaluate_model(
    model=best_cat,
    X=X_train,
    y=Y_train,
    cv=skf,
    scoring_dict=score
)

save_results(
    results_dict=results_cat,
    model_name='CatBoost'
)

save_model(
    model=best_cat,
    model_name='CatBoost'
)

Fitting 5 folds for each of 35 candidates, totalling 175 fits
Лучшие параметры CatBoost: {'random_strength': 3, 'min_data_in_leaf': 10, 'learning_rate': 0.02, 'l2_leaf_reg': 1, 'iterations': 800, 'grow_policy': 'SymmetricTree', 'depth': 3, 'border_count': 32, 'bagging_temperature': 1}
Лучшая ROC_AUC: 0.8824
Результаты CatBoost сохранены
Модель CatBoost сохранена


In [4]:
import pandas as pd


print(pd.read_csv('results_all.csv'))

      Model  Accuracy      F1  ROC-AUC  Precision           Saved_Time
0  CatBoost    0.8294  0.7817    0.883     0.7671  2026-05-12 14:13:01
